# Audit Kesalahan Fungsi dan Ground Truth — MASSIVE-Agents

Memeriksa perbedaan prediksi fungsi dengan GT pada **979 record development**. Hasil BFCL AST dari notebook sebelumnya tidak diubah. Audit memisahkan kesalahan model dari kemungkinan ambiguitas atau ketidaksesuaian GT.

**Catatan:** audit manual tidak otomatis mengubah GT atau skor baseline. Data test hanya boleh diperiksa untuk validitas struktur dataset, bukan untuk memilih model atau menyetel metode.

## 1. Konfigurasi dan lokasi data

Notebook ini menggunakan hasil `bfcl_ast_evaluation.ipynb` yang sudah tersimpan. Tidak memerlukan Ollama, internet, atau inferensi ulang.

In [1]:
from pathlib import Path
import csv
import json
import random
from collections import Counter, defaultdict
from datetime import datetime, timezone
from IPython.display import display


def cari_root():
    cwd = Path.cwd().resolve()
    for folder in [cwd, *cwd.parents, Path(r"D:\KP-Agentic-AI")]:
        if (folder / "results" / "bfcl_ast_evaluation" / "ast_per_prediction.csv").exists():
            return folder
    raise FileNotFoundError("Tidak menemukan results/bfcl_ast_evaluation/ast_per_prediction.csv")

ROOT = cari_root()
INPUT = ROOT / "results" / "bfcl_ast_evaluation"
DATA = ROOT / "data" / "processed" / "massive_agents_reviewed"
OUTPUT = ROOT / "results" / "audit_kesalahan_fungsi"
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 20261110
JUMLAH_SAMPEL = 60
print("Project:", ROOT)
print("Output:", OUTPUT)

Project: D:\KP-Agentic-AI
Output: D:\KP-Agentic-AI\results\audit_kesalahan_fungsi


## 2. Membaca hasil evaluasi AST

Pemeriksaan awal memastikan data evaluasi dan daftar salah fungsi berasal dari hasil development yang sama.

In [2]:
def baca_csv(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def simpan_csv(path, rows):
    if not rows:
        return
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


ast_rows = baca_csv(INPUT / "ast_per_prediction.csv")
salah_rows = baca_csv(INPUT / "audit_salah_fungsi.csv")
assert len(ast_rows) == 2937, "Jumlah hasil AST berbeda dari baseline; periksa sumber file."
assert len({(r['row_key'], r['model']) for r in ast_rows}) == 2937
assert all(r['row_key'] in {x['row_key'] for x in ast_rows} for r in salah_rows)
print("Prediksi:", len(ast_rows))
print("Prediksi salah fungsi:", len(salah_rows))
print("Instruksi unik salah fungsi:", len({r['row_key'] for r in salah_rows}))
print("Group unik:", len({r['group_id'] for r in salah_rows}))

Prediksi: 2937
Prediksi salah fungsi: 520
Instruksi unik salah fungsi: 286
Group unik: 196


## 3. Distribusi pola kesalahan

Frekuensi dihitung per **prediksi**, bukan per instruksi. Satu instruksi dapat salah pada lebih dari satu model.

In [3]:
pola = Counter((r['target_function'], r['prediction_function']) for r in salah_rows)
for (target, prediksi), jumlah in pola.most_common(15):
    print(f"{jumlah:>3}  {target} -> {prediksi}")
simpan_csv(OUTPUT / "pola_kesalahan_fungsi_520.csv", [
    {'fungsi_gt': g, 'fungsi_prediksi': p, 'jumlah_prediksi': n}
    for (g,p),n in pola.most_common()
])

 62  calendar.set -> alarm.set
 51  qa.factoid -> qa.definition
 24  calendar.set -> calendar.query
 13  play.radio -> play.music
 12  qa.factoid -> recommendation.locations
 12  news.query -> qa.factoid
  9  recommendation.events -> recommendation.locations
  9  transport.query -> transport.ticket
  9  takeaway.query -> takeaway.order
  9  audio.volume_mute -> general.joke
  8  news.query -> social.query
  8  music.query -> play.music
  8  calendar.query -> alarm.query
  7  qa.factoid -> weather.query
  7  iot.hue_lightup -> iot.hue_lightdim


## 4. Dataset sumber development

Instruksi dan definisi fungsi diambil dari file preprocessing yang sudah disimpan. Data tidak diperbaiki atau disunting otomatis.

In [4]:
sumber = {}
for bahasa in ("en-US", "id-ID"):
    with open(DATA / f"{bahasa}_development.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            sumber[record['row_key']] = record
assert len(sumber) == 979

prediksi_per_row = defaultdict(dict)
for r in ast_rows:
    prediksi_per_row[r['row_key']][r['model']] = r
salah_per_row = defaultdict(list)
for r in salah_rows:
    salah_per_row[r['row_key']].append(r)
assert all(k in sumber for k in salah_per_row)
print("Data development valid:", len(sumber))

Data development valid: 979


## 5. Memilih 60 kasus prioritas

Sampel dipilih **secara purposive dan stratified untuk audit**, bukan sampel acak representatif. Prioritas diberikan pada kasus yang salah pada banyak model, pola confusion yang umum, dan variasi EN–ID. Usahakan satu instruksi per `group_id`. Statistik hasil audit 60 sampel **tidak boleh diperlakukan sebagai prevalensi dari 520 kesalahan**.

In [5]:
# Sampel awal berisi kasus penting dari distribusi kesalahan baseline.
ANCHOR = [
    'en-US:0001067', 'id-ID:0000733', 'id-ID:0000639', 'en-US:0000038',
    'id-ID:0001150', 'en-US:0000844', 'id-ID:0001211', 'id-ID:0001300',
    'id-ID:0000602', 'id-ID:0000713', 'id-ID:0000852', 'id-ID:0000868',
    'id-ID:0000757', 'id-ID:0000260', 'en-US:0000907', 'en-US:0001045',
    'id-ID:0000582', 'en-US:0000926', 'en-US:0000941', 'id-ID:0000439',
]

rng = random.Random(SEED)
terpilih, kunci_dipakai, grup_dipakai = [], set(), set()
bahasa_dipilih = Counter()

def pilih(k):
    if k not in sumber or k not in salah_per_row or k in kunci_dipakai:
        return False
    grup = sumber[k]['group_id']
    if grup in grup_dipakai:
        return False
    terpilih.append(k)
    kunci_dipakai.add(k)
    grup_dipakai.add(grup)
    bahasa_dipilih[sumber[k]['language_code']] += 1
    return True

for k in ANCHOR:
    pilih(k)

for pair, jumlah in pola.most_common(17):
    kandidat = [k for k, rr in salah_per_row.items()
                if any((x['target_function'], x['prediction_function']) == pair for x in rr)]
    rng.shuffle(kandidat)
    kandidat.sort(key=lambda k: (-len(salah_per_row[k]), bahasa_dipilih[sumber[k]['language_code']], k))
    batas = 5 if jumlah >= 35 else 3 if jumlah >= 10 else 2
    sudah = sum(any((x['target_function'], x['prediction_function']) == pair
                    for x in salah_per_row[k]) for k in terpilih)
    for k in kandidat:
        if len(terpilih) >= JUMLAH_SAMPEL or sudah >= batas:
            break
        if pilih(k):
            sudah += 1

sisa = list(salah_per_row)
rng.shuffle(sisa)
sisa.sort(key=lambda k: (bahasa_dipilih[sumber[k]['language_code']] >= 30,
                          -len(salah_per_row[k]),
                          -max(pola[(x['target_function'], x['prediction_function'])] for x in salah_per_row[k]), k))
for k in sisa:
    if len(terpilih) >= JUMLAH_SAMPEL: break
    if bahasa_dipilih[sumber[k]['language_code']] >= 30: continue
    pilih(k)
for k in sisa:
    if len(terpilih) >= JUMLAH_SAMPEL: break
    pilih(k)
assert len(terpilih) == JUMLAH_SAMPEL
print('Sampel:', len(terpilih), dict(bahasa_dipilih))

Sampel: 60 {'en-US': 30, 'id-ID': 30}


## 6. Membuat lembar audit manual

Kolom `status_gt_manual`, `fungsi_alternatif_jika_ambigu`, `alasan_manual`, dan `sumber_verifikasi` harus diisi berdasarkan pemeriksaan manusia. Indikasi berikut adalah **petunjuk pertanyaan audit**, bukan hasil penilaian.

In [6]:
NAMA_MODEL = {
    'qwen': 'qwen2.5:7b-instruct',
    'gemma': 'gemma2:9b-instruct-q4_K_M',
    'sahabat': 'Supa-AI/gemma2-9b-cpt-sahabatai-v1-instruct:q4_k_m'
}
PETUNJUK = {
 ('calendar.set','calendar.query'): 'Apakah meminta mencatat atau melihat agenda?',
 ('calendar.set','alarm.set'): 'Apakah meminta alarm/pengingat atau acara kalender?',
 ('qa.factoid','qa.definition'): 'Apakah meminta fakta spesifik atau definisi?',
 ('qa.factoid','recommendation.locations'): 'Apakah bertanya lokasi faktual atau meminta rekomendasi?',
 ('news.query','qa.factoid'): 'Apakah meminta berita terkini atau fakta umum?',
 ('play.radio','play.music'): 'Apakah meminta radio atau lagu?',
 ('takeaway.query','takeaway.order'): 'Apakah mengecek layanan atau melakukan pemesanan?',
 ('transport.query','transport.ticket'): 'Apakah bertanya rute/jadwal atau membeli tiket?',
}

def ringkas_prediksi(k, nama_model):
    r = prediksi_per_row[k][nama_model]
    return f"{r['prediction_function']} | AST={int(r['bfcl_ast_correct'].lower() == 'true')}"

anotasi = []
for i,k in enumerate(terpilih,1):
    record = sumber[k]
    errors = salah_per_row[k]
    pair = Counter((r['target_function'],r['prediction_function']) for r in errors).most_common(1)[0][0]
    n = len(errors)
    anotasi.append({
        'no':i, 'prioritas':f'P{4-n} - {n} model salah fungsi',
        'row_key':k, 'record_id':record['source_record_id'],
        'bahasa':record['language_code'], 'group_id':record['group_id'],
        'instruksi':record['question'], 'fungsi_gt':pair[0],
        'gt_argumen_json':json.dumps(record['gt'],ensure_ascii=False),
        'qwen_prediksi':ringkas_prediksi(k,NAMA_MODEL['qwen']),
        'gemma_prediksi':ringkas_prediksi(k,NAMA_MODEL['gemma']),
        'sahabat_prediksi':ringkas_prediksi(k,NAMA_MODEL['sahabat']),
        'jumlah_model_salah_fungsi':n,
        'pola_dominan':f'{pair[0]} -> {pair[1]}',
        'petunjuk_pemeriksaan':PETUNJUK.get(pair,'Bandingkan maksud instruksi dan definisi fungsi.'),
        'status_gt_manual':'BELUM_DIREVIEW', 'fungsi_alternatif_jika_ambigu':'',
        'alasan_manual':'', 'sumber_verifikasi':'', 'reviewer':'', 'tanggal_review':''
    })
simpan_csv(OUTPUT/'audit_fungsi_prioritas_60.csv',anotasi)
print('Disimpan:', OUTPUT/'audit_fungsi_prioritas_60.csv')

Disimpan: D:\KP-Agentic-AI\results\audit_kesalahan_fungsi\audit_fungsi_prioritas_60.csv


## 7. Audit integritas skema GT

Audit struktur terhadap seluruh record, termasuk split test, **hanya untuk memastikan nama fungsi dan parameter terdaftar**. Hasil test tidak digunakan memilih model. GT yang valid secara skema belum tentu benar secara semantik.

In [7]:
hasil_skema = []
files = [DATA/f'{bahasa}_{split}.jsonl' for split in ('development','test') for bahasa in ('en-US','id-ID')]
files.append(DATA/'excluded_after_review.jsonl')
for path in files:
    if not path.exists():
        print('File opsional tidak ada:', path.name)
        continue
    with open(path,encoding='utf8') as f:
        for line in f:
            r=json.loads(line); gt=r['gt']; nama=next(iter(gt))
            katalog={t['name']:t['parameters']['properties'] for t in r['function']}
            tidak_ada=sorted(set(gt[nama])-set(katalog.get(nama,{})))
            hasil_skema.append({
                'row_key':r['row_key'], 'split':r['split'], 'review_status':r['review_status'],
                'fungsi_gt':nama, 'fungsi_valid':nama in katalog,
                'parameter_tidak_dikenal':', '.join(tidak_ada),
                'valid_skema':nama in katalog and not tidak_ada})
simpan_csv(OUTPUT/'audit_skema_semua_split.csv',hasil_skema)
print('Jumlah diperiksa:',len(hasil_skema))
print('Tidak valid:',sum(not x['valid_skema'] for x in hasil_skema))
for x in hasil_skema:
    if not x['valid_skema']: print(x['row_key'], x['parameter_tidak_dikenal'])

Jumlah diperiksa: 3287
Tidak valid: 2
en-US:0001212 meal_type
id-ID:0000826 meal_type


## 8. Menyimpan metadata audit

Sampel dipilih untuk pemeriksaan kasus, bukan untuk mengukur prevalensi kesalahan secara statistik. Setelah anotasi selesai, jangan mengubah GT asli; keputusan revisi dataset harus disimpan sebagai versi baru yang terlacak.

In [8]:
meta = {
    'run': 'audit_kesalahan_fungsi_v1', 'created_utc':datetime.now(timezone.utc).isoformat(),
    'input_ast':str(INPUT/'ast_per_prediction.csv'),
    'input_wrong_function':str(INPUT/'audit_salah_fungsi.csv'),
    'split_for_model_audit':'development','predictions':len(ast_rows),
    'wrong_function_predictions':len(salah_rows),
    'wrong_function_unique_records':len(salah_per_row),
    'selected_cases':len(anotasi), 'selection':'purposive + stratified, group_id unique',
    'selection_seed':SEED, 'manual_decisions':'pending',
    'gt_modified':False,'note':'Kategori GT/manual tidak dapat disimpulkan hanya dari prediksi model.'
}
with open(OUTPUT/'metadata_audit.json','w',encoding='utf8') as f:
    json.dump(meta,f,ensure_ascii=False,indent=2)
print('Selesai membuat berkas audit di',OUTPUT)
print('Selanjutnya isi kolom status_gt_manual dan alasan_manual pada file audit, lalu kirim hasilnya.')

Selesai membuat berkas audit di D:\KP-Agentic-AI\results\audit_kesalahan_fungsi
Selanjutnya isi kolom status_gt_manual dan alasan_manual pada file audit, lalu kirim hasilnya.
